# Reviewer-proof semantic HNSW benchmark

This notebook addresses the final reviewer concerns directly. It does **not** invent another search algorithm. It measures the existing live compiled-predicate traversal against a selectivity-aware HNSW over-fetch baseline, using the same graph within each comparison.

Default paper run: **1,000 queries**, **3 predetermined predicate sets**, semantic selectivity **50% → 2%**, and over-fetch multipliers **0.75×, 1×, 1.5×, 2× K/selectivity**. It also benchmarks the exact 384-D normalized-dot kernel used by the custom traversal, so predicate cost can be stated in units of dense distance computations.

Important: Recall here is **traversal recall** — brute-force dense top-K among items passing the same compiled semantic predicate. It is not end-to-end relevance.


In [ ]:
#@title 1) Settings
FULL_DATA = True #@param {type:"boolean"}
QUERIES = 1000 #@param {type:"integer"}
FRACTIONS = '0.50,0.20,0.10,0.05,0.02' #@param {type:"string"}
OVERFETCH_MULTIPLIERS = '0.75,1.0,1.5,2.0' #@param {type:"string"}
RECALL_TOLERANCE = 0.005 #@param {type:"number"}
print({'FULL_DATA': FULL_DATA, 'QUERIES': QUERIES, 'fractions': FRACTIONS, 'overfetch': OVERFETCH_MULTIPLIERS})


In [ ]:
#@title 2) Clone repo + install dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[benchmark]'], check=True)
if shutil.which('rustc') is None or shutil.which('cargo') is None:
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    os.environ['PATH'] = str(pathlib.Path.home()/'.cargo'/'bin') + os.pathsep + os.environ.get('PATH','')
print('commit:', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())
print('python:', sys.version.split()[0])
print('rustc:', subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Run reviewer benchmark
import pathlib, subprocess, sys, os, time
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
OUT = pathlib.Path('/content/semantic_hnsw_reviewer')
cmd = [sys.executable, '-m', 'experiments.semantic_hnsw_reviewer_sweep',
       '--config', CFG, '--output-dir', str(OUT),
       '--queries', str(QUERIES), '--fractions', FRACTIONS,
       '--overfetch-multipliers', OVERFETCH_MULTIPLIERS,
       '--recall-tolerance', str(RECALL_TOLERANCE)]
print(' '.join(cmd))
t0=time.time(); subprocess.run(cmd, check=True)
print(f'finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
#@title 4) Matched-recall result
import pandas as pd, json
matched = pd.read_csv(OUT/'matched_recall.csv')
pairs = pd.read_csv(OUT/'same_run_pairs.csv')
env = json.loads((OUT/'environment.json').read_text())
display(matched.sort_values(['predicate_set','target_fraction'], ascending=[True,False]))
print('CPU:', env['cpu_model'])
print('same-kernel 384D dot ns:', env['same_kernel_384d_dot_ns'])
print('Recall definition:', env['recall_definition'])


In [ ]:
#@title 5) Over-fetch recall / latency frontier
import matplotlib.pyplot as plt
for name, g in pairs.groupby('predicate_set'):
    fig, ax = plt.subplots(figsize=(8,5))
    for frac, q in g.groupby('target_fraction'):
        q=q.sort_values('overfetch_ms')
        ax.plot(q.overfetch_ms, q.overfetch_traversal_recall, marker='o', label=f'overfetch {frac:.0%}')
        ax.scatter([q.live_ms.iloc[0]],[q.live_traversal_recall.iloc[0]], marker='x', s=80)
    ax.set_title(name)
    ax.set_xlabel('Mean latency (ms)')
    ax.set_ylabel('Traversal Recall@50')
    ax.legend()
    ax.grid(True, alpha=.25)
    plt.show()


In [ ]:
#@title 6) Predicate cost in units of one 384-D distance
cost = pairs[['predicate_set','target_fraction','approx_ns_per_predicate_eval','same_kernel_dot_ns','predicate_over_dot']].drop_duplicates()
display(cost.sort_values(['predicate_set','target_fraction'], ascending=[True,False]))
print('Use this ratio in the paper only after inspecting stability across predicate sets/selectivities.')
